# Pattern 3: AgentCore Gateway with IAM Authentication

Add a Gateway layer between agents and the Knowledge Base. The Gateway hides the KB ID
from agents and provides a standard MCP interface. IAM auth controls who can call the Gateway.

**What you get:** KB ID hidden from callers, centralized MCP interface, IAM-based access.

**What you don't get:** Fine-grained per-principal authorization (that's Pattern 4).

## Prerequisites

- Run [Pattern 1](01-direct-sdk.ipynb) first — it creates the shared bucket, uploads the
  sample documents, and creates the KB execution role. (The setup cell here re-runs it
  idempotently, and additionally creates the **Gateway role**.)
- IAM permissions for Bedrock, AgentCore (`bedrock-agentcore-control`), S3, and IAM.

## Architecture

```
Agent (IAM) ──► Gateway (MCP) ──► KB Target ──► Managed KB ──► S3
                 │
                 └── KB ID is hidden; agent only sees the Gateway URL
```

In [ ]:
import boto3
import time
import json
import util   # util.py in this folder — shared bucket + upload + roles

# --- Configuration ---
REGION = "us-west-2"
S3_BUCKET = "<existing-or-unique-name-for-your-kb-bucket->"
S3_PREFIX = "documents/"

session = boto3.Session()

# Reuse the SAME bucket + docs + KB execution role as Pattern 1 (idempotent),
# then create the Gateway role the AgentCore Gateway assumes to retrieve.
info = util.setup(
    bucket_name=S3_BUCKET,
    prefix=S3_PREFIX,
    metadata=util.SAMPLE_FILE_METADATA,
    region_name=REGION,
)
ROLE_ARN    = info["role_arn"]
S3_BUCKET   = info["bucket"]
S3_PREFIX   = info["prefix"]
GW_ROLE_ARN = util.create_gateway_role(region_name=REGION)

# Clients
cp = session.client("bedrock-agent", region_name=REGION)
dp = session.client("bedrock-agent-runtime", region_name=REGION)
ac = session.client("bedrock-agentcore-control", region_name=REGION)
S3_ACCOUNT = session.client("sts").get_caller_identity()["Account"]

print(f"boto3 {boto3.__version__}")
print(f"KB role:      {ROLE_ARN}")
print(f"Gateway role: {GW_ROLE_ARN}")


In [ ]:
# Create KB + Data Source + Ingest (same as Pattern 1)
response = cp.create_knowledge_base(
    name=f"p3-gateway-{int(time.time())}",
    roleArn=ROLE_ARN,
    knowledgeBaseConfiguration={
        "type": "MANAGED",
        "managedKnowledgeBaseConfiguration": {}   # empty = managed default embedding
    }
)
kb_id = response["knowledgeBase"]["knowledgeBaseId"]
print(f"KB: {kb_id}")

for _ in range(30):
    if cp.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"] == "ACTIVE":
        break
    time.sleep(5)
print("KB ACTIVE")

response = cp.create_data_source(
    knowledgeBaseId=kb_id,
    name="s3-source",
    dataSourceConfiguration={
        "type": "MANAGED_KNOWLEDGE_BASE_CONNECTOR",
        "managedKnowledgeBaseConnectorConfiguration": {
            "connectorParameters": {
                "type": "S3",
                "version": "1",
                "connectionConfiguration": {
                    "bucketName": S3_BUCKET,
                    "bucketOwnerAccountId": S3_ACCOUNT
                },
                "filterConfiguration": {"inclusionPrefixes": [S3_PREFIX]},
                "deletionProtectionConfiguration": {"enableDeletionProtection": False}
            },
            "deletionProtectionConfiguration": {"deletionProtectionStatus": "DISABLED"}
        }
    },
    vectorIngestionConfiguration={
        "parsingConfiguration": {"parsingStrategy": "SMART_PARSING"}
    }
)
ds_id = response["dataSource"]["dataSourceId"]
print(f"DS: {ds_id}")

for _ in range(12):
    if cp.get_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)["dataSource"]["status"] == "AVAILABLE":
        break
    time.sleep(5)
print("DS AVAILABLE")

response = cp.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)
job_id = response["ingestionJob"]["ingestionJobId"]
for _ in range(40):
    job = cp.get_ingestion_job(
        knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id
    )["ingestionJob"]
    if job["status"] in ("COMPLETE", "FAILED"):
        break
    time.sleep(15)

print(f"Ingestion: {job['status']}")

In [ ]:
# Create Gateway with IAM authentication
gw_response = ac.create_gateway(
    name=f"p3-gw-{int(time.time())}",
    roleArn=GW_ROLE_ARN,
    protocolType="MCP",
    authorizerType="AWS_IAM"
)

gw_id = gw_response["gatewayId"]
print(f"Gateway: {gw_id}")

# Wait for READY
gw_url = None
for _ in range(24):
    gw = ac.get_gateway(gatewayIdentifier=gw_id)
    if gw["status"] == "READY":
        gw_url = gw.get("gatewayUrl", "N/A")
        break
    time.sleep(5)

print(f"Status: {gw['status']}")
print(f"Gateway URL: {gw_url}")

In [ ]:
# Create KB Target — connects the Gateway to the Knowledge Base
# Uses the bedrock-knowledge-bases connector
target_response = ac.create_gateway_target(
    gatewayIdentifier=gw_id,
    name="kb-retrieve",
    description="Retrieve from managed KB via Gateway",
    targetConfiguration={
        "mcp": {
            "connector": {
                "source": {"connectorId": "bedrock-knowledge-bases"},
                "configurations": [{
                    "name": "Retrieve",
                    # Tool description exposed to the agent over MCP — this is what
                    # the LLM reads to decide when to call this KB.
                    "description": (
                        "Search two corporate documents: (1) Octank Financial's 10-K annual "
                        "report — financial statements, asset/liability schedules, exhibits, and "
                        "investor disclosures; and (2) a U.S. tornado background & forecasting "
                        "report — where tornadoes form, annual frequency (~1,200/yr), and NOAA data."
                    ),
                    "parameterValues": {
                        "knowledgeBaseId": kb_id,
                        "retrievalConfiguration": {
                            "managedSearchConfiguration": {
                                "numberOfResults": 5
                            }
                        }
                    }
                }]
            }
        }
    },
    credentialProviderConfigurations=[
        {"credentialProviderType": "GATEWAY_IAM_ROLE"}
    ]
)

target_id = target_response["targetId"]
print(f"Target: {target_id}")

# Wait for READY
for _ in range(12):
    t = ac.get_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
    if t["status"] == "READY":
        break
    time.sleep(5)
print(f"Target status: {t['status']}")


In [ ]:
# Verify: direct SDK retrieve still works
response = dp.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={"text": "What are the company's key financial results and risks?"},
    retrievalConfiguration={
        "managedSearchConfiguration": {"numberOfResults": 3}
    }
)

results = response.get("retrievalResults", [])
for i, res in enumerate(results, 1):
    print(f"  {i}. score={res['score']:.4f} | {res['content']['text'][:80]}")
print(f"Total: {len(results)} results")

In [ ]:
# Call the KB THROUGH the gateway using the official MCP client (SigV4-signed).
# Unlike dp.retrieve() — which hits Bedrock directly by kb_id and bypasses the
# gateway — this goes to gw_url, so the gateway's auth/routing actually applies.
#
# An AWS_IAM gateway requires every request to be SigV4-signed. The MCP client
# talks over httpx, which has no built-in SigV4, so we wrap botocore's signer in
# a small httpx.Auth and hand it to the client via `http_client=`. The client
# then performs the initialize() handshake and SSE transport for us.
import httpx
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


class SigV4HTTPXAuth(httpx.Auth):
    """httpx auth handler that SigV4-signs each request for the bedrock-agentcore service."""
    requires_request_body = True   # we must see the body to sign it

    def __init__(self, credentials, service, region):
        self._credentials, self._service, self._region = credentials, service, region

    def auth_flow(self, request):
        aws_req = AWSRequest(
            method=request.method,
            url=str(request.url),
            data=request.content,
            headers=dict(request.headers),
        )
        SigV4Auth(self._credentials, self._service, self._region).add_auth(aws_req)
        request.headers.update(dict(aws_req.headers))   # copy the signed headers back
        yield request


sigv4 = SigV4HTTPXAuth(session.get_credentials(), "bedrock-agentcore", REGION)

async with httpx.AsyncClient(auth=sigv4) as http_client:
    async with streamable_http_client(gw_url, http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session_mcp:
            await session_mcp.initialize()                # MCP handshake (required)

            # 1. Ask the gateway what tools it exposes (what an agent sees over MCP).
            listing = await session_mcp.list_tools()
            print(f"Gateway tools: {[t.name for t in listing.tools]}")
            tool = next(t for t in listing.tools if t.name.split("___")[-1] == "Retrieve")
            print(f"Using tool:   {tool.name}")
            print(f"Description:  {(tool.description or '')[:120]}...")

            # 2. Call that tool THROUGH the gateway — the KB ID is never sent by us.
            result = await session_mcp.call_tool(
                name=tool.name,
                arguments={"retrievalQuery": {
                    "text": "What are the company's key financial results and risks?"
                }},
            )
            print(f"\nisError: {result.isError}")
            print("=== Retrieved via gateway ===")
            print(result.content[0].text[:800] if result.content else result)

## What the Gateway Adds

| Without Gateway (Pattern 1) | With Gateway (Pattern 3) |
|---|---|
| Agent knows the KB ID | KB ID is hidden — only the Gateway knows it |
| Agent calls Bedrock API directly | Agent calls Gateway URL (MCP protocol) |
| No centralized access point | Single Gateway URL for all agents |
| IAM policy per KB | IAM policy on the Gateway |

The Gateway acts as a reverse proxy: agents connect to the Gateway URL, and the Gateway
routes requests to the configured KB target. The agent never sees the KB ID.

**Next:** [Pattern 4](04-gateway-cedar.ipynb) adds Cedar policies for per-principal authorization.

In [ ]:
# Cleanup — order matters: target → gateway → data source → KB
ac.delete_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
time.sleep(3)
ac.delete_gateway(gatewayIdentifier=gw_id)
cp.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)
cp.delete_knowledge_base(knowledgeBaseId=kb_id)
print(f"Deleted: Gateway {gw_id}, KB {kb_id}")